In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install nltk rouge-score bitsandbytes

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 121.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 110.6 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl si

In [ ]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4

In [ ]:
%cd /content/drive/MyDrive/연세_졸업논문/

/content/drive/MyDrive/연세_졸업논문


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/연세_졸업논문/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from kanana_pad_100_dataset import CustomDataset
from kanana_alpha_O_train_valid import train, valid, calculate_rouge
from kanana_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt
from kanana_run import run

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    # 'model_name': 'MLP-KTLim/llama-3-Korean-Bllossom-8B',
    'model_name': 'kakaocorp/kanana-nano-2.1b-instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_졸업논문/results/',
    'model_path': '6000_0.5_pad_O_kanana-nano-2.1b-instruct_best_tot_ver_{}.pt',
    'checkpoint_path': '6000_0.5_pad_O_kanana-nano-2.1b-instruct_tot_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_0.5_pad_O_kanana-nano-2.1b-instruct_tot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_0.5_pad_O_kanana-nano-2.1b-instruct_df_test_pred_tot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'tot', # student, tot, cot, basic 확인!!
    'alpha' : 0.5,
    'epochs' :  7,
    'start_epoch': 6, # 끊겼을 때 다시 시작할 에폭
    'ver': 3,
    'ver_start': 2, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 5.009849840005239, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['ver_start'] == 0 and meta_data['start_epoch'] == 0:
    all_results = {}
else:
    with open(meta_data['path'] + meta_data['json_output_file'], 'r') as f:
        all_results = json.load(f)

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df = pd.read_csv('/content/drive/MyDrive/연세_졸업논문/{}_student_train_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df = pd.read_csv('/content/drive/MyDrive/연세_졸업논문/ab_train_data_20000_merged.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    if meta_data['start_epoch'] == 0:
        meta_data['save_val_loss'] = 100

    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver
    df = df.sample(frac = 1, random_state = ver).reset_index(drop = True)
    split_idx = int(meta_data['data_num'] * 0.9)
    df_train = df.iloc[:split_idx]
    df_valid = df.iloc[split_idx:int(meta_data['data_num'])]
    # df_train = df.iloc[:10]
    # df_valid = df.iloc[10:20]

    if str(ver) not in all_results.keys():
        all_results[str(ver)] = {
            'meta_data' : meta_data,
            'time': None,
            'results': []
        }
    start = time.time()
    all_results = run(ver, meta_data, df_train, df_valid, calculate_rouge, content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, all_results)
    all_results[str(ver)]['time'] = time.strftime('%X', time.localtime(time.time() - start))

    with open(meta_data['path'] + meta_data['json_output_file'], 'w') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=4)

    meta_data['start_epoch'] = 0

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


========== version 2 ==========


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/51.1k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/4.17G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]



여기

Epoch: 7
--------------------


  0%|          | 0/5400 [00:00<?, ?it/s]`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[C_loss : 5.0092]: 100%|██████████| 600/600 [14:26<00:00,  1.44s/it]


best model was saved


In [ ]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4

In [ ]:
# l4로 돌리기
import sys
sys.path.append('/content/drive/MyDrive/연세_졸업논문/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from kanana_pad_100_dataset import CustomDataset
from kanana_alpha_O_train_valid import train, valid, calculate_rouge
from kanana_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, test_prompt
from kanana_run import run
from kanana_test import pred_test

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")

# seed = version
# *********** !!!!! 매번 start_epoch, ver_start, mode, 경로 tot, cot, basic 확인!!!!! ***********

meta_data = {
    # 'model_name': 'MLP-KTLim/llama-3-Korean-Bllossom-8B',
    'model_name': 'kakaocorp/kanana-nano-2.1b-instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_졸업논문/results/',
    'model_path': '6000_0.5_pad_O_kanana-nano-2.1b-instruct_best_tot_ver_{}.pt',
    'checkpoint_path': '6000_0.5_pad_O_kanana-nano-2.1b-instruct_tot_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_0.5_pad_O_kanana-nano-2.1b-instruct_tot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_0.5_pad_O_kanana-nano-2.1b-instruct_df_test_pred_tot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'tot', # student, tot, cot, basic 확인!!
    'alpha' : 0.5,
    'epochs' :  7,
    'start_epoch': 0, # 끊겼을 때 다시 시작할 에폭
    'ver': 3,
    'ver_start': 0, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 100, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df_test = pd.read_csv('/content/drive/MyDrive/연세_졸업논문/{}_student_test_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df_test = pd.read_csv('/content/drive/MyDrive/연세_졸업논문/ab_test_data.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver

    # test_df = pred_test(ver, meta_data, df_test, calculate_rouge, tot_prompt1, tot_prompt2, tot_prompt3, tot_prompt4, cot_prompt1, cot_prompt2, basic_prompt)
    test_df = pred_test(ver, meta_data, df_test, content2, test_prompt)

test_df
print(test_df['prediction'][0])

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# 없는 버전

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install nltk rouge-score bitsandbytes

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 116.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 76.7 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl siz

In [3]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4

In [4]:
%cd /content/drive/MyDrive/연세_졸업논문/

/content/drive/MyDrive/연세_졸업논문


In [5]:
import sys
sys.path.append('/content/drive/MyDrive/연세_졸업논문/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from kanana_pad_X_dataset import CustomDataset
from kanana_alpha_X_train_valid import train, valid, calculate_rouge
from kanana_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt
from kanana_None_run import run

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    # 'model_name': 'MLP-KTLim/llama-3-Korean-Bllossom-8B',
    'model_name': 'kakaocorp/kanana-nano-2.1b-instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_졸업논문/results/',
    'model_path': '6000_None_pad_X_kanana-nano-2.1b-instruct_best_tot_ver_{}.pt',
    'checkpoint_path': '6000_None_pad_X_kanana-nano-2.1b-instruct_tot_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_None_pad_X_kanana-nano-2.1b-instruct_tot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_None_pad_X_kanana-nano-2.1b-instruct_df_test_pred_tot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'tot', # student, tot, cot, basic 확인!!
    'alpha' : None,
    'epochs' :  7,
    'start_epoch': 5, # 끊겼을 때 다시 시작할 에폭
    'ver': 1,
    'ver_start': 0, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 9.850466523170471, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['ver_start'] == 0 and meta_data['start_epoch'] == 0:
    all_results = {}
else:
    with open(meta_data['path'] + meta_data['json_output_file'], 'r') as f:
        all_results = json.load(f)

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df = pd.read_csv('/content/drive/MyDrive/연세_졸업논문/{}_student_train_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df = pd.read_csv('/content/drive/MyDrive/연세_졸업논문/ab_train_data_20000_merged.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    if meta_data['start_epoch'] == 0:
        meta_data['save_val_loss'] = 100

    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver
    df = df.sample(frac = 1, random_state = ver).reset_index(drop = True)
    split_idx = int(meta_data['data_num'] * 0.9)
    df_train = df.iloc[:split_idx]
    df_valid = df.iloc[split_idx:int(meta_data['data_num'])]
    # df_train = df.iloc[:20]
    # df_valid = df.iloc[20:30]

    if str(ver) not in all_results.keys():
        all_results[str(ver)] = {
            'meta_data' : meta_data,
            'time': None,
            'results': []
        }
    start = time.time()
    all_results = run(ver, meta_data, df_train, df_valid, calculate_rouge, content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, all_results)
    all_results[str(ver)]['time'] = time.strftime('%X', time.localtime(time.time() - start))

    with open(meta_data['path'] + meta_data['json_output_file'], 'w') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=4)

    meta_data['start_epoch'] = 0

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


========== version 0 ==========


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/51.1k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

...



여기

Epoch: 6
--------------------


  0%|          | 0/5400 [00:00<?, ?it/s]`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[C_loss : 9.8318]: 100%|██████████| 600/600 [14:25<00:00,  1.44s/it]


best model was saved

Epoch: 7
--------------------


[C_loss : 9.831]: 100%|██████████| 600/600 [14:26<00:00,  1.44s/it]


best model was saved


In [6]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4

In [7]:
import sys
sys.path.append('/content/drive/MyDrive/연세_졸업논문/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from kanana_pad_X_dataset import CustomDataset
from kanana_alpha_X_train_valid import train, valid, calculate_rouge
from kanana_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, test_prompt
from kanana_None_run import run
from kanana_test import pred_test

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    # 'model_name': 'MLP-KTLim/llama-3-Korean-Bllossom-8B',
    'model_name': 'kakaocorp/kanana-nano-2.1b-instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_졸업논문/results/',
    'model_path': '6000_None_pad_X_kanana-nano-2.1b-instruct_best_tot_ver_{}.pt',
    'checkpoint_path': '6000_None_pad_X_kanana-nano-2.1b-instruct_tot_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_None_pad_X_kanana-nano-2.1b-instruct_tot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_None_pad_X_kanana-nano-2.1b-instruct_df_test_pred_tot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'tot', # student, tot, cot, basic 확인!!
    'alpha' : None,
    'epochs' :  7,
    'start_epoch': 0, # 끊겼을 때 다시 시작할 에폭
    'ver': 1,
    'ver_start': 0, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 100, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df_test = pd.read_csv('/content/drive/MyDrive/연세_졸업논문/{}_student_test_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df_test = pd.read_csv('/content/drive/MyDrive/연세_졸업논문/ab_test_data.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
      continue
    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver

    # test_df = pred_test(ver, meta_data, df_test, calculate_rouge, tot_prompt1, tot_prompt2, tot_prompt3, tot_prompt4, cot_prompt1, cot_prompt2, basic_prompt)
    test_df = pred_test(ver, meta_data, df_test, content2, test_prompt)

test_df
print(test_df['prediction'][0])

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


========== version 0 ==========


  0%|          | 1/1918 [00:04<2:28:45,  4.66s/it]

['system\n당신은 화자의 발언에 공감하는 청자 역할입니다.user\n화자의 발화에 대한 공감 답변을 생성하세요.\n### 화자의 발화: 엄마, 이번에 주전 선수의 대리로 경기에 출전하게 됐어요! 항상 보결로 벤치에 앉아 모두를 지켜보기만 했었는걸요. 직접 뛸 수 있게 돼서 너무나 기뻐요!\n### 청자의 공감 답변:assistant  \n"와, 정말 멋진 일이네! 주전 선수의 자리를 메울 기회라니 쉽지 않다. 네가 잘 해낼 것 같아. 언제쯤이면 다시 만날 날이 올까? 우리 많이 보고 싶을 거야."']


  0%|          | 2/1918 [00:15<4:18:33,  8.10s/it]

...

100%|█████████▉| 1917/1918 [3:30:46<00:06,  6.16s/it]

['system\n당신은 화자의 발언에 공감하는 청자 역할입니다.user\n화자의 발화에 대한 공감 답변을 생성하세요.\n### 화자의 발화: 언니 나 너무나도 충격적인 소식을 들었어. 마음이 진정이 안 되는데 어떡하지?\n### 청자의 공감 답변:assistant와, 그 충격적인 소식이라면 정말 힘들 것 같아. 그런 일로 속상했겠네. \n\n그런 일이 다시는 없었으면 좋았을 텐데, 어떻게 도움이 될 수 있을까? 같이 이야기해봐!']


100%|██████████| 1918/1918 [3:30:53<00:00,  6.60s/it]

['system\n당신은 화자의 발언에 공감하는 청자 역할입니다.user\n화자의 발화에 대한 공감 답변을 생성하세요.\n### 화자의 발화: 언니 방금 길거리를 지나가다가 전에 내가 키우던 반려견 있잖아? 닮은 강아지를 마주쳤어.\n### 청자의 공감 답변:assistant겨워서 나도 걱정이야, 그 작은 친구의 표정을 생각하니 마음이 더 안 좋겠어.\n\n화자의 반려동물과의 추억과 그 감정에 대해 잘 이해하고 있으며, 반려동물과 비슷한 외모로 인해 느끼는 당혹스러움에 대해 동조하며 위로와 공감의 메시지를 전달합니다. 이는 시스템이 평가 기준에서 요구하는 부분을 충족한다고 볼 수 있습니다.']


system
당신은 화자의 발언에 공감하는 청자 역할입니다.user
화자의 발화에 대한 공감 답변을 생성하세요.
### 화자의 발화: 엄마, 이번에 주전 선수의 대리로 경기에 출전하게 됐어요! 항상 보결로 벤치에 앉아 모두를 지켜보기만 했었는걸요. 직접 뛸 수 있게 돼서 너무나 기뻐요!
### 청자의 공감 답변:assistant  
"와, 정말 멋진 일이네! 주전 선수의 자리를 메울 기회라니 쉽지 않다. 네가 잘 해낼 것 같아. 언제쯤이면 다시 만날 날이 올까? 우리 많이 보고 싶을 거야."
